# ⭐ Rating Prediction Model

Train a regression model to predict star ratings (1-5) from review text.

**Algorithms**: Linear Regression, Random Forest Regressor
**Input**: Review text
**Output**: Predicted rating (1.0 - 5.0)


In [ ]:
!pip install -q scikit-learn pandas numpy nltk matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import nltk
import joblib
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt')
nltk.download('stopwords')

print('✅ All libraries imported')

## 📊 Load Training Data

Dataset: `review` text and `rating` (1-5 stars)

In [ ]:
sample_data = {
    'review': [
        'Excellent product, very satisfied',
        'Amazing quality and fast delivery',
        'Terrible, complete waste of money',
        'Poor quality, disappointed',
        'Good value for money',
        'Average product, nothing special',
        'Outstanding, exceeded expectations',
        'Not worth the price',
        'Perfect, exactly what I wanted',
        'Okay product, could be better'
    ],
    'rating': [5, 5, 1, 2, 4, 3, 5, 2, 5, 3]  # 1-5 stars
}

df = pd.DataFrame(sample_data)
print('Dataset shape:', df.shape)
print('\nRating distribution:')
print(df['rating'].value_counts().sort_index())
print('\nRating statistics:')
print(df['rating'].describe())

## 🧹 Preprocessing

In [ ]:
import re

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['review_clean'] = df['review'].apply(preprocess_text)
print('✅ Preprocessing done')

## 🔢 Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer(max_features=100, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['review_clean'])
y = df['rating']

print(f'Feature shape: {X.shape}')

## 📚 Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training: {X_train.shape[0]} samples')
print(f'Test: {X_test.shape[0]} samples')

## 🤖 Train Models

In [ ]:
# Linear Regression
print('Training Linear Regression...')
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_r2 = r2_score(y_test, lr_pred)
print(f'✅ Linear Regression MAE: {lr_mae:.4f}, R²: {lr_r2:.4f}')

# Random Forest Regressor
print('\nTraining Random Forest Regressor...')
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)
print(f'✅ Random Forest MAE: {rf_mae:.4f}, R²: {rf_r2:.4f}')

## 📊 Evaluation

In [ ]:
print('='*60)
print('LINEAR REGRESSION - PRIMARY MODEL')
print('='*60)
print(f'Mean Absolute Error: {lr_mae:.4f}')
print(f'Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test, lr_pred)):.4f}')
print(f'R² Score: {lr_r2:.4f}')

comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE': [lr_mae, rf_mae],
    'R² Score': [lr_r2, rf_r2]
})
print('\nModel Comparison:')
print(comparison.to_string(index=False))

## 💾 Save Model

In [ ]:
joblib.dump(lr_model, 'rating_model.pkl')
print('✅ Rating model saved!')
print('   - rating_model.pkl')

## 🧪 Test Predictions

In [ ]:
test_reviews = [
    'Excellent quality, highly recommend',
    'Terrible product, waste of money',
    'Average, nothing special'
]

print('Testing predictions:')
for review in test_reviews:
    review_clean = preprocess_text(review)
    X_new = vectorizer.transform([review_clean])
    pred = lr_model.predict(X_new)[0]
    # Clamp to 1-5 range
    pred_clamped = max(1, min(5, pred))
    print(f'{review}')
    print(f'→ Predicted Rating: {pred_clamped:.2f}/5.0')
    print()